# Extract SCEPTER Outputs

This notebook starts after `05_run_scepter_scenarios.ipynb`. It reads the SCEPTER execution manifest, extracts per-run summary files when they exist, joins results back to spatial model units, and writes clean result tables for mapping and MRV accounting.

If notebook `05` only staged runs and did not execute SCEPTER, this notebook still produces a useful status table showing which summaries are missing. Once SCEPTER is connected, rerun this notebook to parse the completed outputs.


## Workflow

1. **Load execution manifest** from `data/scepter_runs/data/outputs/scepter_execution_manifest.csv`.
2. **Extract per-run summaries** from `data/scepter_runs/data/outputs/{run_id}/{run_id}_summary.csv`.
3. **Flag missing results** for staged or failed runs.
4. **Write flat result tables** for downstream analysis.
5. **Join parsed results back to model-unit geometries** from `data/scepter_runs/inputs/scepter_model_units.gpkg`.
6. **Summarize by scenario** so we can compare baseline and ERW application rates.
7. **Export spatial result layers** for maps and reports.

Expected summary CSV formats are either one-row wide tables, or two-column `metric,value` tables. The parser supports both.


In [ ]:
from pathlib import Path
import os
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

def mount_google_drive_if_colab() -> None:
    try:
        from google.colab import drive
    except ModuleNotFoundError:
        return

    drive.mount("/content/drive")


mount_google_drive_if_colab()

COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/erw_spatial_mrv")
COLAB_DATA_ROOT = COLAB_PROJECT_ROOT / "data"
LOCAL_PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def has_erw_package(project_root: Path) -> bool:
    return (project_root / "src" / "erw_mrv" / "__init__.py").exists()


def source_root_candidates() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates = [LOCAL_PROJECT_ROOT, COLAB_PROJECT_ROOT]
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "erw_spatial_mrv"))
    candidates.extend(
        Path(path)
        for path in (
            "/content/erw_spatial_mrv",
            "/content/enhanced_rock_weathering/erw_spatial_mrv",
            "/content/drive/MyDrive/erw_spatial_mrv",
        )
    )
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique


def find_source_project_root() -> Path:
    for candidate in source_root_candidates():
        if has_erw_package(candidate):
            return candidate
    checked = chr(10).join(f"- {candidate}" for candidate in source_root_candidates())
    raise ModuleNotFoundError(
        "Could not find src/erw_mrv. The data can live in Google Drive, but "
        "the notebook still needs the project source folder containing src/erw_mrv. "
        "In Colab, upload/sync the full erw_spatial_mrv project or run from a "
        "checkout that includes src/. Checked: "
        f"{checked}"
    )


SOURCE_PROJECT_ROOT = find_source_project_root()
PROJECT_ROOT = COLAB_PROJECT_ROOT if COLAB_DATA_ROOT.exists() else SOURCE_PROJECT_ROOT
DATA_ROOT = COLAB_DATA_ROOT if COLAB_DATA_ROOT.exists() else PROJECT_ROOT / "data"
os.environ["ERW_MRV_DATA_ROOT"] = str(DATA_ROOT)

SRC = SOURCE_PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SOURCE_PROJECT_ROOT = {SOURCE_PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")

from erw_mrv.paths import SCEPTER_INPUTS, SCEPTER_OUTPUTS, ensure_dir
from erw_mrv.scepter import (
    extract_scepter_results,
    join_results_to_units,
    load_execution_manifest,
    summarize_scepter_results,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)


## Configure Paths


In [ ]:
MANIFEST_PATH = SCEPTER_OUTPUTS / "scepter_execution_manifest.csv"
MODEL_UNITS_PATH = SCEPTER_INPUTS / "scepter_model_units.gpkg"
RESULTS_DIR = ensure_dir(SCEPTER_OUTPUTS / "extracted")

RESULTS_TABLE_PATH = RESULTS_DIR / "scepter_results_long.csv"
SCENARIO_SUMMARY_PATH = RESULTS_DIR / "scepter_scenario_summary.csv"
SPATIAL_RESULTS_PATH = RESULTS_DIR / "scepter_results_by_model_unit.gpkg"

MANIFEST_PATH, MODEL_UNITS_PATH, RESULTS_DIR


## Load Execution Manifest

The manifest tells us which runs were staged or executed and where their output folders live.


In [ ]:
manifest = load_execution_manifest(MANIFEST_PATH)
print(f"Runs in manifest: {len(manifest):,}")
manifest.groupby(["scenario_id", "status"]).size().rename("run_count").reset_index()


## Extract Per-Run Results

For each run, this looks for the expected summary file: `{output_dir}/{run_id}_summary.csv`.


In [ ]:
results = extract_scepter_results(manifest)
results.to_csv(RESULTS_TABLE_PATH, index=False)

print(f"Parsed runs: {(results['result_status'] == 'parsed').sum():,}")
print(f"Missing summaries: {(results['result_status'] == 'missing_summary').sum():,}")
results[["run_id", "scenario_id", "execution_status", "result_status", "summary_path"]].head(20)


## Result Status QA

If all rows are `missing_summary`, notebook `05` probably staged configs but did not run SCEPTER yet. That is fine for now; this table becomes populated after external model execution is enabled.


In [ ]:
status_table = results.groupby(["scenario_id", "execution_status", "result_status"]).size().rename("run_count").reset_index()
status_table


## Scenario Summary

When numeric metrics are parsed from SCEPTER summaries, this table aggregates them by scenario. Until real output files exist, it reports parsed run counts only.


In [ ]:
scenario_summary = summarize_scepter_results(results)
scenario_summary.to_csv(SCENARIO_SUMMARY_PATH, index=False)
scenario_summary


## Join Results Back To Model Units

This creates a spatial layer with one feature per model-unit/scenario result row. It is the bridge from process-model output back to MRV maps.


In [ ]:
spatial_results = join_results_to_units(results, MODEL_UNITS_PATH)
spatial_results.to_file(SPATIAL_RESULTS_PATH, layer="scepter_results", driver="GPKG")

print(f"Spatial result rows: {len(spatial_results):,}")
SPATIAL_RESULTS_PATH


## Quick Map Preview

This preview shows the model units that have result rows. Once CO2 removal metrics are parsed, change `MAP_COLUMN` to that metric.


In [ ]:
MAP_COLUMN = None
candidate_columns = [
    "co2_removal_t",
    "co2_removal_t_ha",
    "cumulative_co2_removal_t",
    "alkalinity_mol_ha",
]
for column in candidate_columns:
    if column in spatial_results.columns:
        MAP_COLUMN = column
        break

fig, ax = plt.subplots(figsize=(10, 8))
if MAP_COLUMN:
    spatial_results.plot(column=MAP_COLUMN, ax=ax, legend=True, cmap="viridis", linewidth=0.1, edgecolor="black")
    ax.set_title(f"SCEPTER results: {MAP_COLUMN}")
else:
    spatial_results.plot(ax=ax, color="#8ecae6", edgecolor="#023047", linewidth=0.2)
    ax.set_title("SCEPTER result rows by model unit/scenario")
ax.set_axis_off()
plt.show()


## Baseline Comparison Placeholder

The main MRV quantity will be **additional CO2 removal**, calculated as each ERW scenario minus the `baseline_no_erw` result for the same model unit. This cell prepares that comparison when a CO2 metric appears in parsed SCEPTER outputs.


In [ ]:
CO2_METRIC_CANDIDATES = [
    "co2_removal_t",
    "cumulative_co2_removal_t",
    "co2_removed_t",
    "net_co2_removal_t",
]
co2_metric = next((column for column in CO2_METRIC_CANDIDATES if column in results.columns), None)

if co2_metric is None:
    additionality = pd.DataFrame(
        columns=["model_unit_id", "scenario_id", "baseline_co2_t", "scenario_co2_t", "additional_co2_t"]
    )
    print("No CO2 removal metric found yet. Rerun after SCEPTER summaries include a CO2 metric.")
else:
    baseline = results[results["scenario_id"] == "baseline_no_erw"][["model_unit_id", co2_metric]].rename(
        columns={co2_metric: "baseline_co2_t"}
    )
    scenarios = results[results["scenario_id"] != "baseline_no_erw"][["model_unit_id", "scenario_id", co2_metric]].rename(
        columns={co2_metric: "scenario_co2_t"}
    )
    additionality = scenarios.merge(baseline, on="model_unit_id", how="left")
    additionality["additional_co2_t"] = additionality["scenario_co2_t"] - additionality["baseline_co2_t"]

additionality_path = RESULTS_DIR / "scepter_additionality.csv"
additionality.to_csv(additionality_path, index=False)
additionality.head(), additionality_path


## Outputs From This Notebook

This notebook writes extracted SCEPTER outputs under `data/scepter_runs/data/outputs/extracted/`:

- `scepter_results_long.csv`: one row per run with execution status and parsed metrics when available.
- `scepter_scenario_summary.csv`: scenario-level summary statistics for parsed numeric metrics.
- `scepter_results_by_model_unit.gpkg`: spatial model-unit/scenario result layer for mapping.
- `scepter_additionality.csv`: ERW scenario minus baseline CO2 removal, written once CO2 metrics are present.

The next workflow step is spatial reporting and uncertainty: map the result layers, summarize removal by AOI/district/scenario, and then vary uncertain assumptions such as runoff, particle size, application rate, and soil chemistry.
